# Train and Compare Multiple Models

This notebook trains multiple models on your dataset and compares them with consistent metrics.

Supported flow:
- Regression dataset (`target_return` present): `xgboost`, `lstm`, `cnn`, `transformer`
- Classification dataset (`label` present): `xgboost_classifier` (+ optional sklearn baselines)

Default dataset path points to `../../data/train_dataset_btcusdt_h30.csv`.

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
)

# Ensure notebook can import modules from ml_pipeline/.
NOTEBOOK_DIR = Path.cwd()
PIPELINE_DIR = NOTEBOOK_DIR.parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from common import FEATURE_COLUMNS
from lstm_model import build_sequence_model

try:
    from xgboost import XGBRegressor, XGBClassifier
except Exception:
    XGBRegressor = None
    XGBClassifier = None

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except Exception:
    torch = None

pd.set_option("display.max_columns", 200)
np.random.seed(42)

In [ ]:
DATASET_CSV = Path("../../data/train_dataset_btcusdt_h30.csv")
TEST_SIZE = 0.2
RANDOM_STATE = 42
SEQ_LEN = 60
EPOCHS = 8
BATCH_SIZE = 128
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.1
LEARNING_RATE = 1e-3

assert DATASET_CSV.exists(), f"Dataset not found: {DATASET_CSV}"
ds = pd.read_csv(DATASET_CSV)

if "target_return" in ds.columns:
    TASK = "regression"
elif "label" in ds.columns:
    TASK = "classification"
else:
    raise ValueError("Dataset must contain either 'target_return' or 'label'")

missing_features = [c for c in FEATURE_COLUMNS if c not in ds.columns]
if missing_features:
    raise ValueError(f"Missing required feature columns: {missing_features}")

print(f"Dataset: {DATASET_CSV}")
print(f"Rows: {len(ds)}")
print(f"Task: {TASK}")
display(ds.head(3))

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(math.sqrt(mean_squared_error(y_true, y_pred)))
    denom = np.maximum(np.abs(y_true), 1e-8)
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)))
    dir_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 else 0.0
    if not np.isfinite(corr):
        corr = 0.0
    r2 = float(r2_score(y_true, y_pred))
    return {
        "mae": mae,
        "rmse": rmse,
        "mape": mape,
        "directional_accuracy": dir_acc,
        "corr": corr,
        "r2": r2,
    }


def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def make_sequence_dataset(ds_reg: pd.DataFrame, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    prices = ds_reg["price"].to_numpy(dtype=float)
    y = ds_reg["target_return"].to_numpy(dtype=float)
    if len(prices) < seq_len + 2:
        raise ValueError(f"Need at least {seq_len + 2} rows, got {len(prices)}")

    log_returns = np.log(np.maximum(prices[1:], 1e-12) / np.maximum(prices[:-1], 1e-12))
    X_list: list[np.ndarray] = []
    y_list: list[float] = []
    for i in range(seq_len, len(ds_reg)):
        seq = log_returns[i - seq_len : i]
        if len(seq) != seq_len:
            continue
        if np.isfinite(seq).all() and np.isfinite(y[i]):
            X_list.append(seq.reshape(seq_len, 1))
            y_list.append(float(y[i]))

    X = np.asarray(X_list, dtype=np.float32)
    y_out = np.asarray(y_list, dtype=np.float32)
    mean = float(X.mean())
    std = float(X.std() or 1.0)
    X = (X - mean) / std
    return X, y_out

In [ ]:
results: list[dict[str, float | str | int]] = []

if TASK == "regression":
    reg_df = ds.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURE_COLUMNS + ["target_return"]).copy()
    X = reg_df[FEATURE_COLUMNS].to_numpy(dtype=float)
    y = reg_df["target_return"].to_numpy(dtype=float)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=False
    )

    if XGBRegressor is not None:
        xgb = XGBRegressor(
            objective="reg:squarederror",
            n_estimators=400,
            learning_rate=0.03,
            max_depth=5,
            min_child_weight=20,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
        xgb.fit(X_train, y_train)
        pred = xgb.predict(X_test)
        row = {"model": "xgboost", **regression_metrics(y_test, pred)}
        row.update({"train_rows": len(X_train), "test_rows": len(X_test)})
        results.append(row)
    else:
        print("xgboost not available, skipping xgboost regressor")

else:
    cls_df = ds.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURE_COLUMNS + ["label"]).copy()
    cls_df["label"] = cls_df["label"].astype(str).str.strip().str.lower()

    labels = [c for c in ["down", "neutral", "up"] if c in set(cls_df["label"]) ]
    if not labels:
        labels = sorted(cls_df["label"].unique().tolist())
    label_to_id = {label: i for i, label in enumerate(labels)}

    X = cls_df[FEATURE_COLUMNS].to_numpy(dtype=float)
    y = cls_df["label"].map(label_to_id).to_numpy(dtype=int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=False
    )

    if XGBClassifier is not None:
        xgb_cls = XGBClassifier(
            objective="multi:softprob",
            n_estimators=400,
            learning_rate=0.03,
            max_depth=5,
            min_child_weight=20,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=1,
            num_class=len(labels),
        )
        xgb_cls.fit(X_train, y_train)
        pred = xgb_cls.predict(X_test)
        row = {"model": "xgboost_classifier", **classification_metrics(y_test, pred)}
        row.update({"train_rows": len(X_train), "test_rows": len(X_test)})
        results.append(row)
    else:
        print("xgboost not available, skipping classifier")

In [ ]:
if TASK == "regression":
    if torch is None:
        print("torch not available, skipping lstm/cnn/transformer")
    else:
        reg_seq_df = ds.replace([np.inf, -np.inf], np.nan).dropna(subset=["price", "target_return"]).reset_index(drop=True)
        X_seq, y_seq = make_sequence_dataset(reg_seq_df, seq_len=SEQ_LEN)

        X_train, X_test, y_train, y_test = train_test_split(
            X_seq, y_seq, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=False
        )

        for model_type in ["lstm", "cnn", "transformer"]:
            torch.manual_seed(RANDOM_STATE)
            model = build_sequence_model(
                model_type=model_type,
                input_dim=int(X_train.shape[-1]),
                hidden_size=HIDDEN_SIZE,
                num_layers=NUM_LAYERS,
                dropout=DROPOUT,
                nhead=4,
            )

            optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
            loss_fn = nn.SmoothL1Loss()
            loader = DataLoader(
                TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                batch_size=BATCH_SIZE,
                shuffle=True,
            )

            model.train()
            for epoch in range(1, EPOCHS + 1):
                losses = []
                for xb, yb in loader:
                    optimizer.zero_grad(set_to_none=True)
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    losses.append(float(loss.item()))
                if epoch in {1, EPOCHS}:
                    print(f"{model_type} epoch={epoch} loss={np.mean(losses):.8f}")

            model.eval()
            with torch.no_grad():
                pred = model(torch.from_numpy(X_test)).numpy()

            row = {"model": model_type, **regression_metrics(y_test, pred)}
            row.update({"train_rows": len(X_train), "test_rows": len(X_test)})
            results.append(row)
else:
    print("Sequence models are only used in regression mode in this notebook.")

In [ ]:
results_df = pd.DataFrame(results)
if results_df.empty:
    raise RuntimeError("No models were trained; check package availability and dataset schema.")

if TASK == "regression":
    results_df = results_df.sort_values(["rmse", "mae"], ascending=True).reset_index(drop=True)
else:
    results_df = results_df.sort_values(["accuracy", "macro_f1"], ascending=False).reset_index(drop=True)

display(results_df)
print(f"Best model: {results_df.iloc[0]['model']}")

In [ ]:
import matplotlib.pyplot as plt

if TASK == "regression":
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    results_df.plot(kind="bar", x="model", y="rmse", legend=False, ax=ax[0], color="steelblue")
    results_df.plot(kind="bar", x="model", y="directional_accuracy", legend=False, ax=ax[1], color="seagreen")
    ax[0].set_title("RMSE (lower is better)")
    ax[1].set_title("Directional Accuracy (higher is better)")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    results_df.plot(kind="bar", x="model", y="accuracy", legend=False, ax=ax[0], color="darkorange")
    results_df.plot(kind="bar", x="model", y="macro_f1", legend=False, ax=ax[1], color="mediumpurple")
    ax[0].set_title("Accuracy (higher is better)")
    ax[1].set_title("Macro F1 (higher is better)")

for a in ax:
    a.tick_params(axis="x", rotation=20)
    a.set_xlabel("Model")

plt.tight_layout()
plt.show()

In [ ]:
metrics_out = Path("../../data/model_compare_metrics.csv")
metrics_out.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(metrics_out, index=False)
print(f"Saved metrics: {metrics_out}")

best = results_df.iloc[0].to_dict()
display(pd.DataFrame([best]))